# C03 回测闭环与 RQAlpha 模板
前两节课我们已经解决了两个问题：
- 有什么数据可以用
- 数据里能提什么特征

这一节开始回答第三个问题：

> 如果我真的按照某个规则交易，账户净值会怎么变化？

所以本节重点不是“最好的策略”，而是“最小可用回测闭环”。


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG", "600519.XSHG"]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


## 1) 回测闭环到底是什么
一个最简单的回测框架，至少要包含四件事：
1. 什么时候调仓
2. 每次买什么、买多少
3. 持有期间收益怎么计算
4. 如何把每一期收益累积成净值曲线

上一届 `L07` 的核心，就是把这四件事拆开实现。


### 1.1 为什么这节课不急着上复杂策略
因为对零基础同学来说，回测最容易学偏：
- 看了很多净值图
- 但不知道净值是怎么一步步算出来的

这节课的目标，是把“信号 -> 权重 -> 收益 -> 净值”这条链路讲清楚。


In [ ]:
# 这里选几只宽基ETF做演示，更适合讲“组合”和“调仓”
assets = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "513100.XSHG"]
close = rqdatac.get_price(
    assets,
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["close"],
    adjust_type="pre",
    expect_df=True,
).reset_index()

close = close.sort_values(["order_book_id", "date"]).copy()
close["ret"] = close["close"] / close.groupby("order_book_id")["close"].shift(1) - 1
close.head()


In [ ]:
# 转成宽表之后，月度调仓和横截面排名都会更直观
close_wide = close.pivot(index="date", columns="order_book_id", values="close")
ret_wide = close.pivot(index="date", columns="order_book_id", values="ret")
close_wide.head()


## 2) 先定义一个足够简单的策略信号
课堂上保留一个最小版本：
- 每个月月底看一次
- 计算过去 20 个交易日动量
- 选动量最高的 2 个 ETF
- 等权持有到下一个调仓日

这条规则足够简单，但能把整个回测流程走通。


In [ ]:
# 20日动量：今天价格 / 20天前价格 - 1
momentum_20 = close_wide / close_wide.shift(20) - 1
rebalance_dates = pd.DatetimeIndex(momentum_20.resample("ME").last().index)
momentum_20.loc[rebalance_dates].head()


In [ ]:
# 每个调仓日选动量最高的 2 个标的，并做等权配置
weights = pd.DataFrame(0.0, index=rebalance_dates, columns=momentum_20.columns)

for dt in rebalance_dates:
    score = momentum_20.loc[dt].dropna().sort_values(ascending=False)
    top2 = score.head(2).index.tolist()
    if top2:
        weights.loc[dt, top2] = 1 / len(top2)

weights.head()


### 2.1 权重表为什么是回测里的关键中间层
很多同学会从“信号”直接跳到“净值”，但中间其实少了一层：
- 信号告诉你买谁
- 权重告诉你买多少

真正的回测执行，更像是“按权重更新组合”，而不是“凭印象下单”。


In [ ]:
# 把月度权重扩展成日频权重：调仓日更新，其余日期沿用上一次持仓
daily_weights = weights.reindex(ret_wide.index).ffill().fillna(0)

# 组合当日收益 = 前一日持仓权重 * 当日资产收益
portfolio_ret = (daily_weights.shift(1) * ret_wide).sum(axis=1).fillna(0)
nav = (1 + portfolio_ret).cumprod()
nav.head()


## 3) 加一点现实交易摩擦
真实交易不会只有“理想收益”，还会有：
- 手续费
- 滑点
- 调仓时的换手成本

本节先加最简单的一层：按持仓变化量估算交易费用。


In [ ]:
# 用持仓变化量近似换手，再乘上费率
turnover = daily_weights.diff().abs().sum(axis=1).fillna(0)
fee_rate = 0.001
portfolio_ret_after_fee = portfolio_ret - turnover * fee_rate
nav_after_fee = (1 + portfolio_ret_after_fee).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
nav.plot(ax=ax, label="无费用")
nav_after_fee.plot(ax=ax, label="含费用")
ax.set_title("最小回测闭环：净值对比")
ax.legend()
plt.show()


In [ ]:
# 输出最基础的绩效表：累计收益、年化收益、年化波动、夏普
perf_table = pd.Series({
    "cum_return": nav_after_fee.iloc[-1] - 1,
    "annual_return": nav_after_fee.iloc[-1] ** (252 / len(nav_after_fee)) - 1,
    "annual_vol": portfolio_ret_after_fee.std() * (252 ** 0.5),
    "sharpe": (portfolio_ret_after_fee.mean() * 252) / (portfolio_ret_after_fee.std() * (252 ** 0.5)),
}).round(4)

perf_table


### 3.1 这两张结果到底在告诉我们什么
这里最想让学生记住两点：
1. 回测结果不是“直接生成”的，而是从持仓和收益逐步累积出来的
2. 交易成本通常会显著改变净值结果，所以不能只看“理想化收益”


## 4) RQAlpha 模板：把研究逻辑搬进回测框架
手写回测的意义，是帮助大家理解计算过程。

如果以后想把策略交给框架跑，就需要把逻辑放进固定入口：
- `init`
- `before_trading`
- `handle_bar`
- `scheduler`

这也是上一届 `L08` 的重点。


In [ ]:
# 下面这段是模板，不要求这节课现场一定跑通。
# 它的作用是帮助大家把“研究环境里的逻辑”映射到 RQAlpha 结构里。

# from rqalpha import run_func

# def init(context):
#     context.assets = ["510300.XSHG", "510500.XSHG", "159915.XSHE", "513100.XSHG"]
#
# def handle_bar(context, bar_dict):
#     # 这里会放入计算信号、更新权重、下单的逻辑
#     pass
#
# config = {...}
# result = run_func(init=init, handle_bar=handle_bar, config=config)


## 5) 课上小练习


### 练习 1：自己改成 top1 策略
要求：
1. 把上面的 `top2` 改成 `top1`
2. 重新得到 `weights` 和 `nav`
3. 对比 top1 和 top2 的净值曲线


In [ ]:
# 练习 1：学生现场自己写
# 提示：只需要修改选取标的的数量，其余逻辑基本不变


In [ ]:
# 参考答案
weights_top1 = pd.DataFrame(0.0, index=rebalance_dates, columns=momentum_20.columns)
for dt in rebalance_dates:
    score = momentum_20.loc[dt].dropna().sort_values(ascending=False)
    top1 = score.head(1).index.tolist()
    if top1:
        weights_top1.loc[dt, top1] = 1.0

daily_weights_top1 = weights_top1.reindex(ret_wide.index).ffill().fillna(0)
ret_top1 = (daily_weights_top1.shift(1) * ret_wide).sum(axis=1).fillna(0)
nav_top1 = (1 + ret_top1).cumprod()
nav_top1.tail()


### 练习 2：自己加手续费
要求：
1. 基于 `daily_weights` 计算换手 `turnover`
2. 设定费率 `0.0015`
3. 得到含手续费的净值序列


In [ ]:
# 练习 2：学生现场自己写


In [ ]:
# 参考答案
turnover_ex2 = daily_weights.diff().abs().sum(axis=1).fillna(0)
fee_rate_ex2 = 0.0015
ret_ex2 = portfolio_ret - turnover_ex2 * fee_rate_ex2
nav_ex2 = (1 + ret_ex2).cumprod()
nav_ex2.tail()


## 小结
这一节的关键词是：**回测闭环**。

到这里为止，大家应该能回答四个问题：
1. 调仓日是怎么定的
2. 权重表是怎么生成的
3. 组合收益如何由单资产收益加总得到
4. 交易成本为什么会显著影响结果

本节常见易错点：
1. 只有信号，没有权重
2. 直接用当日权重乘当日收益，忽略了“先持仓、后发生收益”的时序
3. 完全忽略交易成本
